In [13]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix
from numpy.linalg import solve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

In [14]:
# 1. Load the dataset from the file path
case_name = 'pglib_opf_case14_ieee'
total_samples = 10000
dataset_path = f'./dataset/{case_name}_{total_samples}.pt'
problem = torch.load(dataset_path)

# 2. Determine the exact sizes for the splits
actual_total_samples = problem["Pd_all"].shape[0] 
train_size = int(0.8 * actual_total_samples)
val_size = int(0.1 * actual_total_samples)
test_size = actual_total_samples - train_size - val_size

print(f"Loaded {actual_total_samples} samples. Splitting into -> Train: {train_size}, Val: {val_size}, Test: {test_size}")

# 3. Slice the tensors for each split
# --- Training Data (First 80%) ---
train_Pd = problem["Pd_all"][:train_size]
train_Qd = problem["Qd_all"][:train_size]

# --- Validation Data (Next 10%) ---
val_Pd = problem["Pd_all"][train_size : train_size + val_size]
val_Qd = problem["Qd_all"][train_size : train_size + val_size]

# --- Testing Data (Final 10%) ---
test_Pd = problem["Pd_all"][train_size + val_size :]
test_Qd = problem["Qd_all"][train_size + val_size :]

# 4. Setup the PyTorch DataLoader for the training split
batch_size = 32
train_dataset = TensorDataset(train_Pd, train_Qd)

# shuffle=True randomizes the batch order every epoch during training
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# (Optional) You can also create DataLoaders for val and test if needed later
# val_dataset = TensorDataset(val_Pd, val_Qd)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

Loaded 10000 samples. Splitting into -> Train: 8000, Val: 1000, Test: 1000


# Base line model

## 1. PINN Architecture

In [22]:
class baselineQCQPMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]
    Output:
        v:  [B, 2*nbus] (Rectangular voltages)
        pg: [B, ngen]   (Active generation)
        qg: [B, ngen]   (Reactive generation)
    """
    def __init__(self, nbus: int, ngen: int, slack_imag_idx: int, hidden: int = 512):
        super().__init__()
        self.nbus = nbus
        self.ngen = ngen
        self.in_dim = 2 * nbus
        self.out_dim_v = 2 * nbus
        self.out_dim_g = 2 * ngen 
        self.slack_imag_idx = int(slack_imag_idx)

        # Core MLP
        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, self.out_dim_v + self.out_dim_g),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, problem: dict) -> tuple:
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)
        raw = self.net(x)

        # 1. Slice outputs
        v_raw = raw[:, :self.out_dim_v]
        g_raw = raw[:, self.out_dim_v:]
        
        pg_raw = g_raw[:, :self.ngen]
        qg_raw = g_raw[:, self.ngen:]

        # 2. Bound Voltages to [-Vmax, Vmax] using Tanh for smooth gradients
        Vmax_b = problem["Vmax"].reshape(1, -1).expand(B, -1)
        Vmax_full = torch.cat([Vmax_b, Vmax_b], dim=-1) # For real and imaginary parts
        v = torch.tanh(v_raw) * Vmax_full

        # Constraint (2m): Enforce slack imaginary part = 0 exactly
        v_clone = v.clone()
        v_clone[:, self.slack_imag_idx] = 0.0
        v = v_clone

        # 3. Bound Generation strictly between [min, max] using Sigmoid
        pmax_b = problem["pmax"].reshape(1, -1).expand(B, -1)
        pmin_b = problem["pmin"].reshape(1, -1).expand(B, -1)
        qmax_b = problem["qmax"].reshape(1, -1).expand(B, -1)
        qmin_b = problem["qmin"].reshape(1, -1).expand(B, -1)

        pg = pmin_b + torch.sigmoid(pg_raw) * (pmax_b - pmin_b)
        qg = qmin_b + torch.sigmoid(qg_raw) * (qmax_b - qmin_b)

        return v, pg, qg

## 2. The QCQP Loss Function

In [23]:
def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_qcqp_loss(model: nn.Module, Pd_batch: torch.Tensor, Qd_batch: torch.Tensor, problem: dict, weights: dict):
    B = Pd_batch.shape[0]
    
    # Predict continuous variables (bounded by construction)
    v, pg, qg = model(Pd_batch, Qd_batch, problem)

    # --------------------------------------------------------
    # Unpack Problem Matrices
    # --------------------------------------------------------
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"] # [nbus, ngen]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    
    # Expand Generator Limits for batch
    pmax = problem["pmax"].unsqueeze(0).expand(B, -1)
    pmin = problem["pmin"].unsqueeze(0).expand(B, -1)
    qmax = problem["qmax"].unsqueeze(0).expand(B, -1)
    qmin = problem["qmin"].unsqueeze(0).expand(B, -1)

    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1)

    # --------------------------------------------------------
    # Evaluate Quadratic Forms
    # --------------------------------------------------------
    vp = quad_batch_stack(v, M_p)   # [B, nbus]
    vq = quad_batch_stack(v, M_q)
    
    pf = quad_batch_stack(v, M_pf)  # 2b
    qf = quad_batch_stack(v, M_qf)  # 2c
    pt = quad_batch_stack(v, M_pt)  # 2d
    qt = quad_batch_stack(v, M_qt)  # 2e
    
    vc = quad_batch_stack(v, M_c)
    vs = quad_batch_stack(v, M_s)
    vv = quad_batch_stack(v, M_v)

    # --------------------------------------------------------
    # Constraints Mapping (Equations 2h - 2m)
    # --------------------------------------------------------
    
    # 1. Objective (Eq 2a) - Minimize total active generation
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()
    
    # 2. Branch Thermal Limits (Eq 2f & 2g): p^2 + q^2 <= smax^2
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2
    
    # 3. Nodal Power Balance (Eq 2h & 2i): [C_g * pg]_i - pd_i = v^T M_p v
    # pg @ C_g.T automatically maps the ngen generations to their nbus locations
    h_p = (pg @ C_g.T) - Pd_batch - vp
    h_q = (qg @ C_g.T) - Qd_batch - vq
    # Generator Limits
    g_pg_max = pg - pmax
    g_pg_min = pmin - pg
    g_qg_max = qg - qmax
    g_qg_min = qmin - qg

    # 4. Angle Difference Stability (Eq 2j & 2k)
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc

    # 5. Voltage Magnitude Security (Eq 2l): Vmin^2 <= v^T M_v v <= Vmax^2
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv

    # --------------------------------------------------------
    # Compute Penalties
    # --------------------------------------------------------
    loss_eq_p = h_p.pow(2).mean()
    loss_eq_q = h_q.pow(2).mean()
    
    loss_thermal = F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean()
    loss_ang = F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean()
    loss_v = F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean()

    total_loss = (
        weights["eq_p"] * loss_eq_p +
        weights["eq_q"] * loss_eq_q +
        weights["thermal"] * loss_thermal +
        weights["ang"] * loss_ang +
        weights["v"] * loss_v +
        weights["obj"] * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "obj_cost": obj.detach().item(),
        
        "max_h_p": h_p.abs().max().detach().item(),
        "max_h_q": h_q.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "max_v_viol": torch.max(F.relu(g_v_max).max(), F.relu(g_v_min).max()).detach().item(),
        
        # This will consistently report 0.0 in the baseline!
        "max_gen_viol": torch.max(
            torch.max(F.relu(g_pg_max).max(), F.relu(g_pg_min).max()),
            torch.max(F.relu(g_qg_max).max(), F.relu(g_qg_min).max())
        ).detach().item()
    }
    return total_loss, diagnostics

## 3. Training

In [33]:
# --- 0. HARDWARE TUNING FOR INTEL i7-1255U ---
# Explicitly force PyTorch to utilize all 12 logical threads across P-cores and E-cores
torch.set_num_threads(12)
device = torch.device("cpu")

# --- 1. LOAD AND PREP DATA ---
case_name = 'pglib_opf_case14_ieee'
total_samples = 10000
dataset_path = f'./dataset/{case_name}_{total_samples}.pt'

# This loads grid physics (nbus, ngen, a_ref) AND the datasets (Pd_all, Qd_all)
problem = torch.load(dataset_path)

actual_total_samples = problem["Pd_all"].shape[0] 
train_size = int(0.8 * actual_total_samples)

# SLICE AND MOVE DIRECTLY TO MEMORY
# Your 16GB of RAM can easily hold this. Keeping it local prevents memory-transfer bottlenecks.
train_Pd = problem["Pd_all"][:train_size]
train_Qd = problem["Qd_all"][:train_size]

# --- 2. SETUP DATALOADER ---
batch_size = 1024 
train_dataset = TensorDataset(train_Pd, train_Qd)
# num_workers=0 is strictly best here to avoid CPU context-switching overhead on hybrid architectures
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# --- 3. MODEL & OPTIMIZER SETUP ---
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model = baselineQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

loss_weights = {
    "eq_p": 1000.0,
    "eq_q": 1000.0,
    "thermal": 1.0,
    "ang": 1.0,
    "v": 1.0,
    "obj": 0.01
}

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- 4. THE HIGH-SPEED TRAINING LOOP ---
epochs = 200 

for epoch in range(epochs):
    model.train()
    
    # Inner loop now processes the dataset in massive, vectorized chunks
    for Pd_batch, Qd_batch in train_loader:
        
        optimizer.zero_grad()
        loss, diag = compute_qcqp_loss(model, Pd_batch, Qd_batch, problem, loss_weights)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        
    # if epoch % 10 == 0:  
    print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | "
            f"Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | "
            f"Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f}")

Epoch    0 | Cost: 1754.79 | Max P-Miss: 0.8666 | Max Q-Miss: 0.5600 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    1 | Cost: 1451.76 | Max P-Miss: 0.8046 | Max Q-Miss: 0.4418 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    2 | Cost: 1200.31 | Max P-Miss: 0.5602 | Max Q-Miss: 0.3452 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    3 | Cost: 1088.32 | Max P-Miss: 0.5806 | Max Q-Miss: 0.1991 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    4 | Cost: 1097.61 | Max P-Miss: 0.4555 | Max Q-Miss: 0.2276 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    5 | Cost: 1098.64 | Max P-Miss: 0.4339 | Max Q-Miss: 0.1757 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    6 | Cost: 1081.68 | Max P-Miss: 0.3865 | Max Q-Miss: 0.1384 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    7 | Cost: 1081.31 | Max P-Miss: 0.3438 | Max Q-Miss: 0.1687 | Max Gen Viol: 0.0000 | Max Thermal: 0.0000
Epoch    8 | Cost: 1111.80 | Max P-Miss: 0.3297 | Max Q-Miss: 0.1753 | Max Gen Viol: 0.0

In [37]:
print("\nTraining complete. Saving model weights...")

# Define the save path
model_save_path = f"./model/pinn_model_{case_name}_{epochs}epochs.pth"

# Save the state dictionary
torch.save(model.state_dict(), model_save_path)
print(f"Model successfully saved to: {model_save_path}")


Training complete. Saving model weights...
Model successfully saved to: ./model/pinn_model_pglib_opf_case14_ieee_200epochs.pth


In [34]:
print("\n--- INITIATING TEST SET EVALUATION ---")
# 1. Switch model from Training mode to Evaluation mode
model.eval()

# 2. Disable gradient tracking to save memory and increase speed
with torch.no_grad():
    
    # 3. Extract the 10% Test Set (the last 1,000 samples)
    val_size = int(0.1 * actual_total_samples)
    test_Pd = problem["Pd_all"][train_size + val_size:].to(device)
    test_Qd = problem["Qd_all"][train_size + val_size:].to(device)
    
    # 4. Run the entire test set through the model at once
    test_loss, test_diag = compute_qcqp_loss(model, test_Pd, test_Qd, problem, loss_weights)
    
    print("=== FINAL PINN TEST METRICS ===")
    print(f"Average Generation Cost: ${test_diag['obj_cost']:7.2f}")
    print(f"Max Active Power Mismatch: {test_diag['max_h_p']:.6f} p.u.")
    print(f"Max Reactive Power Mismatch: {test_diag['max_h_q']:.6f} p.u.")
    print(f"Max Thermal Limit Violation: {test_diag['max_thermal']:.6f}")
    print(f"Max Generator Limit Violation: {test_diag['max_gen_viol']:.6f}")


--- INITIATING TEST SET EVALUATION ---
=== FINAL PINN TEST METRICS ===
Average Generation Cost: $1458.17
Max Active Power Mismatch: 0.214082 p.u.
Max Reactive Power Mismatch: 0.081027 p.u.
Max Thermal Limit Violation: 0.000000
Max Generator Limit Violation: 0.000000


# PINN paper

## Rahul's Branched Architecture in PyTorch

In [ ]:
class RahulSinglePINN_Smax(nn.Module):
    """
    Version 2: Single Neural Network for all variables (Primal + Dual).
    """
    def __init__(self, nbus, ngen, nbranch, hidden_dim=512):
        super().__init__()
        self.nbus = nbus
        self.ngen = ngen
        self.nbranch = nbranch
        
        in_dim = 2 * nbus # Pd and Qd
        
        # Calculate total output dimension
        self.dim_v = 2 * nbus
        self.dim_g = 2 * ngen # pg and qg
        self.num_duals = (4 * nbus) + (4 * nbranch) + (4 * ngen)
        
        out_dim = self.dim_v + self.dim_g + self.num_duals
        
        # A SINGLE Neural Network for everything
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, Pd, Qd):
        x = torch.cat([Pd, Qd], dim=-1)
        
        # Single forward pass
        raw = self.net(x)
        
        # ----------------------------------------------------
        # 1. Slice Primal Variables (Unbounded)
        # ----------------------------------------------------
        idx = 0
        v = raw[:, idx : idx + self.dim_v]; idx += self.dim_v
        
        pq = raw[:, idx : idx + self.dim_g]; idx += self.dim_g
        pg = pq[:, :self.ngen]
        qg = pq[:, self.ngen:]
        
        # ----------------------------------------------------
        # 2. Slice Dual Variables (Lagrange Multipliers)
        # ----------------------------------------------------
        lam_p = raw[:, idx : idx+self.nbus]; idx += self.nbus
        lam_q = raw[:, idx : idx+self.nbus]; idx += self.nbus
        
        mu_sf = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        mu_st = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        
        mu_ang_max = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        mu_ang_min = raw[:, idx : idx+self.nbranch]; idx += self.nbranch
        
        mu_v_max = raw[:, idx : idx+self.nbus]; idx += self.nbus
        mu_v_min = raw[:, idx : idx+self.nbus]; idx += self.nbus
        
        mu_pg_max = raw[:, idx : idx+self.ngen]; idx += self.ngen
        mu_pg_min = raw[:, idx : idx+self.ngen]; idx += self.ngen
        mu_qg_max = raw[:, idx : idx+self.ngen]; idx += self.ngen
        mu_qg_min = raw[:, idx : idx+self.ngen]; idx += self.ngen
        
        return (v, pg, qg, lam_p, lam_q, mu_sf, mu_st, 
                mu_ang_max, mu_ang_min, mu_v_max, mu_v_min, 
                mu_pg_max, mu_pg_min, mu_qg_max, mu_qg_min)

## Rahul's Primal-Dual KKT Loss Function

In [ ]:
# Helper function to compute (M * v) for batches
def batch_Mv(M: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    # M: [K, d, d], v: [B, d] -> [B, K, d]
    return torch.einsum('kij,bj->bki', M, v)

def compute_rahul_kkt_smax_loss(model, Pd_batch, Qd_batch, problem, weights):
    B = Pd_batch.shape[0]
    
    # Forward Pass
    (v, pg, qg, lam_p, lam_q, mu_sf, mu_st, 
     mu_ang_max, mu_ang_min, mu_v_max, mu_v_min, 
     mu_pg_max, mu_pg_min, mu_qg_max, mu_qg_min) = model(Pd_batch, Qd_batch)

    # Matrices & Limits
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    pmax = problem["pmax"].unsqueeze(0).expand(B, -1)
    pmin = problem["pmin"].unsqueeze(0).expand(B, -1)
    qmax = problem["qmax"].unsqueeze(0).expand(B, -1)
    qmin = problem["qmin"].unsqueeze(0).expand(B, -1)
    
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1)

    # --------------------------------------------------------
    # A. PRIMAL EVALUATIONS
    # --------------------------------------------------------
    vp = quad_batch_stack(v, M_p); vq = quad_batch_stack(v, M_q)
    pf = quad_batch_stack(v, M_pf); qf = quad_batch_stack(v, M_qf)
    pt = quad_batch_stack(v, M_pt); qt = quad_batch_stack(v, M_qt)
    vc = quad_batch_stack(v, M_c); vs = quad_batch_stack(v, M_s)
    vv = quad_batch_stack(v, M_v)

    # Equations
    h_p = (pg @ C_g.T) - Pd_batch - vp
    h_q = (qg @ C_g.T) - Qd_batch - vq
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv
    g_pg_max = pg - pmax; g_pg_min = pmin - pg
    g_qg_max = qg - qmax; g_qg_min = qmin - qg

    # --------------------------------------------------------
    # CALCULATE OBJECTIVE COST
    # --------------------------------------------------------
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()

    # --------------------------------------------------------
    # B. COMPLEMENTARY SLACKNESS (mu * g == 0)
    # --------------------------------------------------------
    cs_loss = (
        (mu_sf * g_sf).pow(2).mean() + (mu_st * g_st).pow(2).mean() +
        (mu_ang_max * g_ang_max).pow(2).mean() + (mu_ang_min * g_ang_min).pow(2).mean() +
        (mu_v_max * g_v_max).pow(2).mean() + (mu_v_min * g_v_min).pow(2).mean() +
        (mu_pg_max * g_pg_max).pow(2).mean() + (mu_pg_min * g_pg_min).pow(2).mean() +
        (mu_qg_max * g_qg_max).pow(2).mean() + (mu_qg_min * g_qg_min).pow(2).mean()
    )

    # --------------------------------------------------------
    # C. DUAL FEASIBILITY (mu >= 0)
    # --------------------------------------------------------
    dual_feas_loss = (
        F.relu(-mu_sf).pow(2).mean() + F.relu(-mu_st).pow(2).mean() +
        F.relu(-mu_ang_max).pow(2).mean() + F.relu(-mu_ang_min).pow(2).mean() +
        F.relu(-mu_v_max).pow(2).mean() + F.relu(-mu_v_min).pow(2).mean() +
        F.relu(-mu_pg_max).pow(2).mean() + F.relu(-mu_pg_min).pow(2).mean() +
        F.relu(-mu_qg_max).pow(2).mean() + F.relu(-mu_qg_min).pow(2).mean()
    )

    # --------------------------------------------------------
    # D. KKT STATIONARITY (Analytical Gradients = 0)
    # --------------------------------------------------------
    dL_dpg = (2 * c2 * pg) + c1 + (lam_p @ C_g) + mu_pg_max - mu_pg_min
    dL_dqg = (lam_q @ C_g) + mu_qg_max - mu_qg_min

    # Exact Analytical Gradient w.r.t voltage (v)
    dL_dv_p = -2 * torch.einsum('bk,bki->bi', lam_p, batch_Mv(M_p, v))
    dL_dv_q = -2 * torch.einsum('bk,bki->bi', lam_q, batch_Mv(M_q, v))
    
    # The Quartic Analytical Gradient for smax: 4 * mu * (p * Mp*v + q * Mq*v)
    dL_dv_sf = 4 * torch.einsum('bk,bk,bki->bi', mu_sf, pf, batch_Mv(M_pf, v)) + \
               4 * torch.einsum('bk,bk,bki->bi', mu_sf, qf, batch_Mv(M_qf, v))
    dL_dv_st = 4 * torch.einsum('bk,bk,bki->bi', mu_st, pt, batch_Mv(M_pt, v)) + \
               4 * torch.einsum('bk,bk,bki->bi', mu_st, qt, batch_Mv(M_qt, v))
    
    dL_dv_vmax = 2 * torch.einsum('bk,bki->bi', mu_v_max, batch_Mv(M_v, v))
    dL_dv_vmin = -2 * torch.einsum('bk,bki->bi', mu_v_min, batch_Mv(M_v, v))
    
    M_s_v = batch_Mv(M_s, v); M_c_v = batch_Mv(M_c, v)
    t_max = torch.tan(angmax).unsqueeze(-1); t_min = torch.tan(angmin).unsqueeze(-1)
    
    dL_dv_angmax = torch.einsum('bk,bki->bi', mu_ang_max, 2 * M_s_v - 2 * t_max * M_c_v)
    dL_dv_angmin = torch.einsum('bk,bki->bi', mu_ang_min, 2 * t_min * M_c_v - 2 * M_s_v)

    dL_dv = dL_dv_p + dL_dv_q + dL_dv_sf + dL_dv_st + dL_dv_vmax + dL_dv_vmin + dL_dv_angmax + dL_dv_angmin
    
    stationarity_loss = dL_dpg.pow(2).mean() + dL_dqg.pow(2).mean() + dL_dv.pow(2).mean()

    # --------------------------------------------------------
    # E. PRIMAL LOSS (Actual Physical Violations)
    # --------------------------------------------------------
    primal_loss = (
        h_p.pow(2).mean() + h_q.pow(2).mean() +
        F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean() +
        F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean() +
        F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean() +
        F.relu(g_pg_max).pow(2).mean() + F.relu(g_pg_min).pow(2).mean() +
        F.relu(g_qg_max).pow(2).mean() + F.relu(g_qg_min).pow(2).mean()
    )

    total_loss = (
        weights["primal"] * primal_loss +
        weights["cs"] * cs_loss +
        weights["dual_feas"] * dual_feas_loss +
        weights["stationarity"] * stationarity_loss
    )

    # --------------------------------------------------------
    # DIAGNOSTICS FOR BENCHMARKING
    # --------------------------------------------------------
    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_primal": primal_loss.detach().item(),
        "loss_kkt_stat": stationarity_loss.detach().item(),
        "loss_kkt_cs": cs_loss.detach().item(),
        
        "obj_cost": obj.detach().item(),
        "max_h_p": h_p.abs().max().detach().item(),
        "max_h_q": h_q.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "max_gen_viol": torch.max(
            torch.max(F.relu(g_pg_max).max(), F.relu(g_pg_min).max()),
            torch.max(F.relu(g_qg_max).max(), F.relu(g_qg_min).max())
        ).detach().item()
    }

    return total_loss, diagnostics

## Training

In [ ]:
model_rahul = RahulSinglePINN_Smax(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    nbranch=problem["nbranch"]
).to(device)

optimizer_rahul = optim.Adam(model_rahul.parameters(), lr=1e-3)

# Note: Stationarity loss can explode easily because of the quartic s_max gradient.
# You may need to tune "stationarity" weight down if the loss goes to NaN.
loss_weights_rahul = {
    "primal": 10.0,         
    "cs": 1.0,              
    "dual_feas": 1.0,       
    "stationarity": 0.01     
}

print("Starting Training: Rahul's KKT PINN (Smax)")

for epoch in range(1000):
    Pd_batch = torch.clamp(gaussian_batch(problem["Pd"], batch_size=32), min=0.0)
    Qd_batch = torch.clamp(gaussian_batch(problem["Qd"], batch_size=32), min=0.0)
    
    optimizer_rahul.zero_grad()
    
    loss, diag = compute_rahul_kkt_smax_loss(
        model=model_rahul, 
        Pd_batch=Pd_batch, 
        Qd_batch=Qd_batch, 
        problem=problem, 
        weights=loss_weights_rahul
    )
    
    loss.backward()
    
    # Critical: Clip gradients to prevent the quartic s_max derivatives from exploding
    torch.nn.utils.clip_grad_norm_(model_rahul.parameters(), 10.0)
    optimizer_rahul.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f}")

# DC3


## The DC3 Loss Function with Correction Loop

In [ ]:
# Helper function (same as before)
def batch_Mv(M: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    return torch.einsum('kij,bj->bki', M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_dc3_qcqp_smax_loss(model, Pd_batch, Qd_batch, problem, weights, corr_steps=10, corr_lr=1e-3):
    B = Pd_batch.shape[0]
    
    # --------------------------------------------------------
    # 1. FORWARD PASS (Network Prediction)
    # --------------------------------------------------------
    # Using your baseline MLP that naturally bounds pg and qg
    v_pred, pg_pred, qg_pred = model(Pd_batch, Qd_batch, problem)

    # Unpack Problem Matrices
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1) if "c0" in problem else 0.0

    # --------------------------------------------------------
    # 2. DC3 CORRECTION PHASE (Inner Optimization Loop)
    # --------------------------------------------------------
    # We clone the predictions and detach them from the neural network graph
    v_c = v_pred.detach().clone().requires_grad_(True)
    pg_c = pg_pred.detach().clone().requires_grad_(True)
    qg_c = qg_pred.detach().clone().requires_grad_(True)
    
    # Define an inner optimizer solely for the correction steps
    optimizer_corr = torch.optim.Adam([v_c, pg_c, qg_c], lr=corr_lr)
    
    with torch.enable_grad():
        for _ in range(corr_steps):
            optimizer_corr.zero_grad()
            
            # Evaluate Physics on the *Correction* variables
            vp_c = quad_batch_stack(v_c, M_p)
            vq_c = quad_batch_stack(v_c, M_q)
            pf_c = quad_batch_stack(v_c, M_pf); qf_c = quad_batch_stack(v_c, M_qf)
            pt_c = quad_batch_stack(v_c, M_pt); qt_c = quad_batch_stack(v_c, M_qt)
            vc_c = quad_batch_stack(v_c, M_c); vs_c = quad_batch_stack(v_c, M_s)
            vv_c = quad_batch_stack(v_c, M_v)
            
            # Constraints
            h_p_c = (pg_c @ C_g.T) - Pd_batch - vp_c
            h_q_c = (qg_c @ C_g.T) - Qd_batch - vq_c
            g_sf_c = (pf_c**2 + qf_c**2) - smax**2
            g_st_c = (pt_c**2 + qt_c**2) - smax**2
            g_ang_min_c = torch.tan(angmin) * vc_c - vs_c
            g_ang_max_c = vs_c - torch.tan(angmax) * vc_c
            g_v_max_c = vv_c - (Vmax**2)
            g_v_min_c = (Vmin**2) - vv_c
            
            # Sum up all violations to create a repair gradient
            viol_loss = (
                h_p_c.pow(2).mean() + h_q_c.pow(2).mean() +
                F.relu(g_sf_c).pow(2).mean() + F.relu(g_st_c).pow(2).mean() +
                F.relu(g_ang_min_c).pow(2).mean() + F.relu(g_ang_max_c).pow(2).mean() +
                F.relu(g_v_max_c).pow(2).mean() + F.relu(g_v_min_c).pow(2).mean()
            )
            
            viol_loss.backward()
            optimizer_corr.step()

    # --------------------------------------------------------
    # 3. STANDARD PRIMAL EVALUATION (On original NN output)
    # --------------------------------------------------------
    vp = quad_batch_stack(v_pred, M_p); vq = quad_batch_stack(v_pred, M_q)
    pf = quad_batch_stack(v_pred, M_pf); qf = quad_batch_stack(v_pred, M_qf)
    pt = quad_batch_stack(v_pred, M_pt); qt = quad_batch_stack(v_pred, M_qt)
    vc = quad_batch_stack(v_pred, M_c); vs = quad_batch_stack(v_pred, M_s)
    vv = quad_batch_stack(v_pred, M_v)

    h_p = (pg_pred @ C_g.T) - Pd_batch - vp
    h_q = (qg_pred @ C_g.T) - Qd_batch - vq
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv

    cost_per_gen = c2 * (pg_pred ** 2) + c1 * pg_pred + c0
    obj = cost_per_gen.sum(dim=1).mean()

    primal_loss = (
        h_p.pow(2).mean() + h_q.pow(2).mean() +
        F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean() +
        F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean() +
        F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean()
    )

    # --------------------------------------------------------
    # 4. DC3 TARGET LOSS
    # --------------------------------------------------------
    # Penalize distance between Neural Network output and the Repaired Target
    dc3_corr_loss = (
        F.mse_loss(v_pred, v_c.detach()) + 
        F.mse_loss(pg_pred, pg_c.detach()) + 
        F.mse_loss(qg_pred, qg_c.detach())
    )

    total_loss = (weights["primal"] * primal_loss) + (weights["obj"] * obj) + (weights["dc3_corr"] * dc3_corr_loss)

    # --------------------------------------------------------
    # DIAGNOSTICS FOR BENCHMARKING
    # --------------------------------------------------------
    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_primal": primal_loss.detach().item(),
        "loss_dc3_corr": dc3_corr_loss.detach().item(),
        "obj_cost": obj.detach().item(),
        
        "max_h_p": h_p.abs().max().detach().item(),
        "max_h_q": h_q.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "max_v_viol": torch.max(F.relu(g_v_max).max(), F.relu(g_v_min).max()).detach().item(),
        "max_gen_viol": 0.0 # Baseline model bounds generators by construction!
    }

    return total_loss, diagnostics

## The DC3 Training Loop

In [ ]:
# Use the exact same model architecture as the Baseline for a fair comparison
# (Extract slack index again if you haven't already)
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model_dc3 = baselineQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer_dc3 = optim.Adam(model_dc3.parameters(), lr=1e-3)

# DC3 relies heavily on the correction target weight
loss_weights_dc3 = {
    "primal": 10.0,       # Soft penalty on constraints
    "obj": 0.01,          # Generation cost weight
    "dc3_corr": 50.0      # Heavy weight pushing predictions towards the repaired targets
}

print("Starting Training: DC3 QCQP (With Unrolled Correction)")

for epoch in range(1000):
    Pd_batch = torch.clamp(gaussian_batch(problem["Pd"], batch_size=32), min=0.0)
    Qd_batch = torch.clamp(gaussian_batch(problem["Qd"], batch_size=32), min=0.0)
    
    optimizer_dc3.zero_grad()
    
    # Notice we pass corr_steps=5. This adds computational overhead but heavily accelerates learning feasibility!
    loss, diag = compute_dc3_qcqp_smax_loss(
        model=model_dc3, 
        Pd_batch=Pd_batch, 
        Qd_batch=Qd_batch, 
        problem=problem, 
        weights=loss_weights_dc3,
        corr_steps=5,     # Adjust between 5-20 based on RAM / Time
        corr_lr=1e-2      # Inner loop learning rate
    )
    
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model_dc3.parameters(), 10.0)
    optimizer_dc3.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f} | DC3 Corr: {diag['loss_dc3_corr']:.4f}")

# FSNet

## The FSNet QCQP Loss Function

In [ ]:
# Helper functions
def batch_Mv(M: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    return torch.einsum('kij,bj->bki', M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_fsnet_qcqp_smax_loss(model, Pd_batch, Qd_batch, problem, weights, seek_steps=5, seek_lr=1e-3):
    B = Pd_batch.shape[0]
    
    # --------------------------------------------------------
    # 1. FORWARD PASS (Initial Guess y_0)
    # --------------------------------------------------------
    v_0, pg_0, qg_0 = model(Pd_batch, Qd_batch, problem)

    # Unpack Problem Matrices
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)
    pmax = problem["pmax"].unsqueeze(0).expand(B, -1)
    pmin = problem["pmin"].unsqueeze(0).expand(B, -1)
    qmax = problem["qmax"].unsqueeze(0).expand(B, -1)
    qmin = problem["qmin"].unsqueeze(0).expand(B, -1)
    c2 = problem["c2"].unsqueeze(0).expand(B, -1)
    c1 = problem["c1"].unsqueeze(0).expand(B, -1)
    c0 = problem["c0"].unsqueeze(0).expand(B, -1) if "c0" in problem else 0.0

    # --------------------------------------------------------
    # 2. FSNET FEASIBILITY SEEKING (Differentiable Inner Loop)
    # --------------------------------------------------------
    # We initialize the seeking variables. We DO NOT detach them.
    v = v_0
    pg = pg_0
    qg = qg_0
    
    for _ in range(seek_steps):
        # A. Evaluate Constraints on Current State
        vp = quad_batch_stack(v, M_p); vq = quad_batch_stack(v, M_q)
        pf = quad_batch_stack(v, M_pf); qf = quad_batch_stack(v, M_qf)
        pt = quad_batch_stack(v, M_pt); qt = quad_batch_stack(v, M_qt)
        vc = quad_batch_stack(v, M_c); vs = quad_batch_stack(v, M_s)
        vv = quad_batch_stack(v, M_v)
        
        h_p = (pg @ C_g.T) - Pd_batch - vp
        h_q = (qg @ C_g.T) - Qd_batch - vq
        g_sf = (pf**2 + qf**2) - smax**2
        g_st = (pt**2 + qt**2) - smax**2
        g_ang_min = torch.tan(angmin) * vc - vs
        g_ang_max = vs - torch.tan(angmax) * vc
        g_v_max = vv - (Vmax**2)
        g_v_min = (Vmin**2) - vv
        g_pg_max = pg - pmax; g_pg_min = pmin - pg
        g_qg_max = qg - qmax; g_qg_min = qmin - qg
        
        # Sum up all violations to create the Feasibility Objective P(y)
        viol_loss = (
            h_p.pow(2).mean() + h_q.pow(2).mean() +
            F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean() +
            F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean() +
            F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean() +
            F.relu(g_pg_max).pow(2).mean() + F.relu(g_pg_min).pow(2).mean() +
            F.relu(g_qg_max).pow(2).mean() + F.relu(g_qg_min).pow(2).mean()
        )
        
        # B. Compute Differentiable Gradients
        # create_graph=True is the defining feature of FSNet! 
        # It allows the main optimizer to backpropagate THROUGH this solver loop.
        grad_v, grad_pg, grad_qg = torch.autograd.grad(
            viol_loss, (v, pg, qg), create_graph=True, retain_graph=True
        )
        
        # C. Take a gradient descent step (moving closer to feasibility)
        v = v - seek_lr * grad_v
        pg = pg - seek_lr * grad_pg
        qg = qg - seek_lr * grad_qg

    # --------------------------------------------------------
    # 3. FINAL TASK LOSS EVALUATION ON \hat{y} (Post-Seeking)
    # --------------------------------------------------------
    # The neural network is graded on how good the final point \hat{y} is.
    vp_f = quad_batch_stack(v, M_p); vq_f = quad_batch_stack(v, M_q)
    pf_f = quad_batch_stack(v, M_pf); qf_f = quad_batch_stack(v, M_qf)
    pt_f = quad_batch_stack(v, M_pt); qt_f = quad_batch_stack(v, M_qt)
    vc_f = quad_batch_stack(v, M_c); vs_f = quad_batch_stack(v, M_s)
    vv_f = quad_batch_stack(v, M_v)

    h_p_f = (pg @ C_g.T) - Pd_batch - vp_f
    h_q_f = (qg @ C_g.T) - Qd_batch - vq_f
    g_sf_f = (pf_f**2 + qf_f**2) - smax**2
    g_st_f = (pt_f**2 + qt_f**2) - smax**2
    g_ang_min_f = torch.tan(angmin) * vc_f - vs_f
    g_ang_max_f = vs_f - torch.tan(angmax) * vc_f
    g_v_max_f = vv_f - (Vmax**2)
    g_v_min_f = (Vmin**2) - vv_f
    g_pg_max_f = pg - pmax; g_pg_min_f = pmin - pg
    g_qg_max_f = qg - qmax; g_qg_min_f = qmin - qg

    # Objective Cost at the feasible point
    cost_per_gen = c2 * (pg ** 2) + c1 * pg + c0
    obj = cost_per_gen.sum(dim=1).mean()

    # Remaining Physical Violations at the feasible point
    final_primal_loss = (
        h_p_f.pow(2).mean() + h_q_f.pow(2).mean() +
        F.relu(g_sf_f).pow(2).mean() + F.relu(g_st_f).pow(2).mean() +
        F.relu(g_ang_min_f).pow(2).mean() + F.relu(g_ang_max_f).pow(2).mean() +
        F.relu(g_v_max_f).pow(2).mean() + F.relu(g_v_min_f).pow(2).mean() +
        F.relu(g_pg_max_f).pow(2).mean() + F.relu(g_pg_min_f).pow(2).mean() +
        F.relu(g_qg_max_f).pow(2).mean() + F.relu(g_qg_min_f).pow(2).mean()
    )

    total_loss = (weights["primal"] * final_primal_loss) + (weights["obj"] * obj)

    # --------------------------------------------------------
    # DIAGNOSTICS FOR BENCHMARKING
    # --------------------------------------------------------
    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_primal": final_primal_loss.detach().item(),
        "obj_cost": obj.detach().item(),
        
        "max_h_p": h_p_f.abs().max().detach().item(),
        "max_h_q": h_q_f.abs().max().detach().item(),
        "max_thermal": torch.max(F.relu(g_sf_f).max(), F.relu(g_st_f).max()).detach().item(),
        "max_v_viol": torch.max(F.relu(g_v_max_f).max(), F.relu(g_v_min_f).max()).detach().item(),
        "max_gen_viol": torch.max(
            torch.max(F.relu(g_pg_max_f).max(), F.relu(g_pg_min_f).max()),
            torch.max(F.relu(g_qg_max_f).max(), F.relu(g_qg_min_f).max())
        ).detach().item()
    }

    return total_loss, diagnostics

In [ ]:
# You can use your identical baseline model architecture!
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model_fsnet = baselineQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer_fsnet = optim.Adam(model_fsnet.parameters(), lr=1e-3)

# Note: Because the network is trained on the POST-seeking point,
# we can usually weigh the objective much higher than standard PINNs!
loss_weights_fsnet = {
    "primal": 10.0,       
    "obj": 1.0,           
}

print("Starting Training: FSNet QCQP (Differentiable Seeking)")

for epoch in range(1000):
    Pd_batch = torch.clamp(gaussian_batch(problem["Pd"], batch_size=32), min=0.0)
    Qd_batch = torch.clamp(gaussian_batch(problem["Qd"], batch_size=32), min=0.0)
    
    optimizer_fsnet.zero_grad()
    
    # Run FSNet with 5 unrolled seeking steps
    # Note: If you get "Out Of Memory" (OOM) errors, lower seek_steps to 2 or 3.
    loss, diag = compute_fsnet_qcqp_smax_loss(
        model=model_fsnet, 
        Pd_batch=Pd_batch, 
        Qd_batch=Qd_batch, 
        problem=problem, 
        weights=loss_weights_fsnet,
        seek_steps=5,     
        seek_lr=1e-2      
    )
    
    loss.backward()
    
    # Clipping is mandatory because second-order gradients can explode
    torch.nn.utils.clip_grad_norm_(model_fsnet.parameters(), 10.0)
    optimizer_fsnet.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cost: {diag['obj_cost']:7.2f} | Max P-Miss: {diag['max_h_p']:.4f} | Max Q-Miss: {diag['max_h_q']:.4f} | Max Gen Viol: {diag['max_gen_viol']:.4f} | Max Thermal: {diag['max_thermal']:.4f}")